## Step 0 
Loading the libraries and defines small helper functions.

In [1]:
"Assessment 2 : Hotel Booking analysis pipeline. Usage:hotel_bookings.csv from. Produce: Number Table(CSV formate), PNG , log.txt. Requires: pandas numpy scipy scikit-learn statsmodels matplotlib "
import sys,os,warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,roc_curve,precision_recall_curve,r2_score,mean_squared_error,mean_absolute_error)
warnings.filterwarnings("ignore")

SRC="hotel_bookings.csv" #original dataset taken from kaggel
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
SEED=1
LOG=open(os.path.join(OUT,"analysis_log.txt"),"w")

def log(*a):
    s="".join(str(x) for x in a)  
    print(s); LOG.write(s + "\n")
def table(df,name, index=True):
    df.to_csv(os.path.join(OUT, name), index=index)
def fig(name):
    plt.tight_layout(); plt.savefig(os.path.join(OUT,name),dpi=200);plt.sho()
    
MONTHS = ["January","February","March","April","May","June","July","August",
          "September","October","November","December"]
MNUM={m:i+1 for i,m in enumerate(MONTHS)}



## Step 1- Clean the data
**Objective:** Removing exact duplicates, zero-guest bookings,zero-night stays, negative or extreme ADR (average daily rate), and undefined values.
**Log**: Log are visible at log file for removed rows
**Quoted in report**: The counts are reported in assessment.

== 1. DATA CLEANING==
Raw rows/cols:(119390, 32)
Raw cancellation rate: 0.370
                                      Step   Rows  Removed
                                  Raw rows 119390        0
        After removing zero-guest bookings  87230    32160
           After removing zero-night stays  86639      591
           After removing exact duplicates  86637        2
             After removing undefined meal  86152      485
After removing undefined segment / channel  86147        5
Cleaned cancellation rate: 0.277
ADR==0 rows kept for Problem 1, excluded from Problem 2:1029
Clean analysis file:86147rows,20columns


In [ ]:
#-----------------------------------------------------------1. CLEAN
raw=pd.read_csv(SRC)
log("== 1. DATA CLEANING==")
log("Raw rows/cols:", raw.shape)
log("Raw cancellation rate: %.3f" % raw.is_canceled.mean())
df = raw.drop_duplicates()
steps=[("Raw rows", len(raw))]
df= df.copy(); df["children"]=df["children"].fillna(0)
guests=df.adults + df.children + df.babies
nights=df.stays_in_weekend_nights + df.stays_in_week_nights
df=df[guests > 0]; steps.append(("After removing zero-guest bookings",len(df)))
nights=df.stays_in_weekend_nights + df.stays_in_week_nights
df =df[nights > 0];
steps.append(("After removing zero-night stays", len(df)))
df=df[(df.adr >=0) & (df.adr <=1000)]; steps.append(("After removing exact duplicates", len(df)))
df = df[df.meal != "Undefined"];    steps.append(("After removing undefined meal", len(df)))
df=df[(df.market_segment !="Undefined") & (df.distribution_channel !="Undefined")]
steps.append(("After removing undefined segment / channel",len(df)))
clean_log=pd.DataFrame(steps, columns=["Step","Rows"])
clean_log["Removed"]=(-clean_log.Rows.diff()).fillna(0).astype(int)
table(clean_log, "table_01_cleaning_log.csv", index=False)
log(clean_log.to_string(index=False))
log("Cleaned cancellation rate: %.3f" % df.is_canceled.mean())
log("ADR==0 rows kept for Problem 1, excluded from Problem 2:",(df.adr==0).sum())

COLS = ["is_canceled","adr","hotel","lead_time","arrival_date_year","arrival_date_month","stays_in_weekend_nights","stays_in_week_nights","adults","children","babies","meal", "market_segment", "distribution_channel", "is_repeated_guest","previous_cancellations", "deposit_type", "customer_type", "reserved_room_type","total_of_special_requests"]
d = df[COLS].copy()
d["month_num"] = d.arrival_date_month.map(MNUM)
d["total_nights"] = d.stays_in_weekend_nights + d.stays_in_week_nights
d.to_csv(os.path.join(OUT, "hotel_clean_20cols.csv"), index=False)
log("Clean analysis file:", d.shape[0], "rows,", len(COLS), "columns")
